# 原始G.17半导体工业生产行：来源结构预检

冻结范围见`plan.json`。项目已保存的原始G.17 Table 1与ALFRED可选版本菜单分开检验；本轮仅检查NAICS3344行及表头，不解析因子数值或读取市场结果。

In [ ]:
from pathlib import Path
import hashlib,json,re,os
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/semiconductor_ip_original_g17_bridge_20261022"
plan=json.loads((card/"plan.json").read_text())
source=root/plan["source_card"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and sha(source/"summary.json")==plan["source_summary_sha256"]
assert sha(source/"verification_manifest.json")==plan["source_verification_manifest_sha256"]
assert sha(source/"publication_inventory.csv")==plan["publication_inventory_sha256"]
summary=json.loads((source/"summary.json").read_text())
assert summary["recorder_id"]==plan["source_recorder_id"]
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE","true")
qlib.init(provider_uri=summary["provider_uri"],region="us",kernels=1)
objects,audit=load_verified_recorder_objects(source,expected_recorder_id=plan["source_recorder_id"],artifacts=[plan["raw_artifact"]],expected_manifest_sha256=plan["source_verification_manifest_sha256"],expected_artifact_sha256={plan["raw_artifact"]:plan["raw_artifact_sha256"]})
raw=objects[plan["raw_artifact"]]
assert len(raw)==plan["expected_original_documents"]==192
inventory=pd.read_csv(source/"publication_inventory.csv")
assert len(inventory)==192
rows=[]
for name,body in sorted(raw.items()):
    assert isinstance(body,bytes) and name.endswith("_sup.txt")
    text=body.decode("utf-8","replace");lines=text.splitlines()
    matches=[]
    for i,line in enumerate(lines[:-1]):
        if "Semiconductor and other electronic" not in line:continue
        next_line=lines[i+1]
        if not ("component" in next_line and re.search(r"\|\s*3344\s*\|",next_line)):continue
        prefix="\n".join(lines[:i]);t1=prefix.rfind("Table 1");t2=prefix.rfind("Table 2")
        section="Table1" if t1>t2 and t1>=0 else "other"
        head="\n".join(lines[max(0,i-20):i]);base=re.search(r"Index,\s*(\d{4})\s*=\s*100",head)
        matches.append({"section":section,"row_sha256":hashlib.sha256((line+"\n"+next_line).encode()).hexdigest(),"base_year":base.group(1) if base else None,"has_sa_header":"Seasonally adjusted" in head,"has_ip_header":"INDUSTRIAL PRODUCTION" in head})
    table1=[m for m in matches if m['section']=='Table1']
    rows.append({"release_file":name,"release_date":name[:8],"matches":len(matches),"Table1_3344_matches":len(table1),"row_sha256":table1[0]["row_sha256"] if len(table1)==1 else None,"index_base_year":table1[0]["base_year"] if len(table1)==1 else None,"SA_header_present":table1[0]["has_sa_header"] if len(table1)==1 else False,"IP_header_present":table1[0]["has_ip_header"] if len(table1)==1 else False})
out=pd.DataFrame(rows)
for k in plan['positive_controls']:
    assert out.loc[out.release_file.eq(k),'Table1_3344_matches'].item()==1
assert set(out.release_date)==set(pd.to_datetime(inventory.release_date).dt.strftime('%Y%m%d'))
out.to_csv(card/"row_presence_audit.csv",index=False)
result={"status":"original_G17_Table1_3344_row_presence_audited","source_recorder_id":plan['source_recorder_id'],"plan_sha256":sha(card/'plan.json'),"original_documents":len(out),"Table1_exact_one_count":int(out.Table1_3344_matches.eq(1).sum()),"Table1_missing_count":int(out.Table1_3344_matches.eq(0).sum()),"Table1_ambiguous_count":int(out.Table1_3344_matches.gt(1).sum()),"SA_header_count":int(out.SA_header_present.sum()),"IP_header_count":int(out.IP_header_present.sum()),"base_year_counts":{str(k):int(v) for k,v in out.index_base_year.fillna('unknown').value_counts().items()},"first_release":out.release_date.min(),"last_release":out.release_date.max(),"missing_release_files":out.loc[out.Table1_3344_matches.eq(0),'release_file'].tolist(),"numeric_feature_values_parsed":0,"market_outcome_reads":0,"new_model_fits":0,"native_accounts":0,"historical_first_receipt_qualified":False,"definition_or_base_continuity_qualified":False,"decision":"row_coverage_only_no_Stage1_input_permission"}
(card/'source_row_audit.json').write_text(json.dumps(result,indent=2)+'\n')
summary_out={"experiment":plan['experiment'],"decision":result['decision'],"objective_achieved":False,"result":"source_row_audit.json","source_recorder_id":plan['source_recorder_id'],"source_archive_sha256":summary['nas_evidence']['sha256'],"new_source_GETs":0,"new_fits":0,"new_accounts":0,"artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":[],"source_recorder_authority":True}}
(card/'summary.json').write_text(json.dumps(summary_out,indent=2)+'\n')
(card/'conclusion.md').write_text(f"# 美联储原始G.17中的半导体工业生产行\n\n只读封存Qlib Recorder中{len(out)}份有日期的原始G.17补充表，不抓取新来源或市场收益。Table 1工业生产NAICS3344精确单行{result['Table1_exact_one_count']}份，缺失{result['Table1_missing_count']}份，重复/歧义{result['Table1_ambiguous_count']}份；2011-01-14和2015-08-14是预先登记的正控。指数基年分布见`source_row_audit.json`，全部逐发布版本结构检查见`row_presence_audit.csv`。\n\n这项工业生产行与旧Table 2产能利用率334412-9不同，ALFRED网页最早可选2015版本不等于美联储原始发布档案最早只有2015。当前只证明原件中有行，尚未逐值解析同版月份、核对基年/定义连续性或首次接收时钟。决定：`row_coverage_only_no_Stage1_input_permission`；0因子值、0拟合、0账户。若结构全覆盖，再另立冻结数值/日历资格计划，不能直接把此结果接入Stage1。\n")
print(json.dumps({k:result[k] for k in ('original_documents','Table1_exact_one_count','Table1_missing_count','Table1_ambiguous_count','base_year_counts')}))

## 中文结论

执行后以`conclusion.md`、`source_row_audit.json`和`row_presence_audit.csv`为准。结构存在不等于数值、时点和预测力合格。